# Gemini による BLS 音声文字起こし

`data/0604data/1回目_右前.wav` を Gemini 3.5 Transcribe で文字起こしし、結果を `outputs/transcription/gemini/` に保存します。

## 起動方法

プロジェクトルートで `uv sync` を実行した後、同じPython 3.12環境からこのノートブックを起動してください。

```bash
uv run jupyter lab
```

API キーは環境変数 `GEMINI_API_KEY` から自動的に読み込みます。ノートブック上での入力は求めません。

```bash
export GEMINI_API_KEY='your-api-key'
```

> **注意:** 実行すると音声が Google の Gemini API にアップロードされます。録音対象者の同意と所属組織の個人情報・研究データ取扱規程を確認してから実行してください。ローカルの `data/` 内のファイルは変更しません。

In [4]:
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """pyproject.toml を目印にプロジェクトルートを探す。"""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        "プロジェクトルートを特定できません。MedWhisper_nachi 内から起動してください。"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
AUDIO_PATH = PROJECT_ROOT / "data" / "0604data" / "1回目_右前.wav"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "transcription" / "gemini"
OUTPUT_PATH = OUTPUT_DIR / f"{AUDIO_PATH.stem}_gemini.txt"
MODEL = "gemini-3.5-transcribe"

if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"入力音声が見つかりません: {AUDIO_PATH}")

print(f"入力: {AUDIO_PATH}")
print(f"出力: {OUTPUT_PATH}")
print(f"モデル: {MODEL}")

入力: /media/dl-box/ADATA SE800/med/MedWhisper_nachi/data/0604data/1回目_右前.wav
出力: /media/dl-box/ADATA SE800/med/MedWhisper_nachi/outputs/transcription/gemini/1回目_右前_gemini.txt
モデル: gemini-3.5-transcribe


In [ ]:
import os

try:
    from google import genai
except ImportError as exc:
    raise ImportError(
        "google-genai がありません。先頭の『起動方法』にある uv コマンドで "
        "JupyterLab を起動し直してください。"
    ) from exc

try:
    api_key = os.environ["GEMINI_API_KEY"]
except KeyError as exc:
    raise RuntimeError(
        "環境変数 GEMINI_API_KEY が設定されていません。"
        "JupyterLab を起動する前に設定してください。"
    ) from exc

client = genai.Client(api_key=api_key)
print("Gemini クライアントを初期化しました。")

In [ ]:
import shutil
import tempfile
import warnings

# 日本語ファイル名が HTTP ヘッダーで問題になる環境を避けるため、
# 読み取り専用の元データを ASCII 名の一時ファイルへコピーして送信する。
audio_mime_types = {
    ".wav": "audio/wav",
    ".mp3": "audio/mp3",
    ".flac": "audio/flac",
    ".m4a": "audio/m4a",
    ".ogg": "audio/ogg",
}
try:
    mime_type = audio_mime_types[AUDIO_PATH.suffix.lower()]
except KeyError as exc:
    raise ValueError(f"未対応の音声形式です: {AUDIO_PATH.suffix}") from exc
uploaded_file = None

try:
    with tempfile.TemporaryDirectory(prefix="medwhisper_gemini_") as temp_dir:
        upload_path = Path(temp_dir) / f"audio_input{AUDIO_PATH.suffix.lower()}"
        shutil.copyfile(AUDIO_PATH, upload_path)
        uploaded_file = client.files.upload(
            file=upload_path,
            config={"mime_type": mime_type},
        )

    interaction = client.interactions.create(
        model=MODEL,
        input=[
            {
                "type": "audio",
                "uri": uploaded_file.uri,
                "mime_type": uploaded_file.mime_type,
            }
        ],
        generation_config={
            "transcription_config": {
                "language_codes": ["ja-JP"],
                "custom_vocabulary": [
                    "一次救命処置",
                    "BLS",
                    "胸骨圧迫",
                    "心肺蘇生",
                    "AED",
                    "意識確認",
                    "呼吸確認",
                    "気道確保",
                    "人工呼吸",
                    "119番通報",
                    "救急車",
                ],
                # 評価対象の言いよどみや反復を消さない逐語モード。
                "mode": {"type": "verbatim"},
            }
        },
    )
    transcript = interaction.output_text.strip()
    if not transcript:
        raise RuntimeError("Gemini API から文字起こし結果が返されませんでした。")
finally:
    # API 上の一時ファイルも処理後に削除する。
    if uploaded_file is not None and uploaded_file.name:
        try:
            client.files.delete(name=uploaded_file.name)
        except Exception as exc:
            warnings.warn(f"Gemini API 上の一時ファイルを削除できませんでした: {exc}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(transcript + "\n", encoding="utf-8")

print(transcript)
print(f"\n保存先: {OUTPUT_PATH}")